In [1]:
print("ok")

ok


In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\End-to-end-Medical-Chatbot-with-LLMs-Langchain-Pinecone-Flask-AWS'

In [5]:
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter



In [8]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [13]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [14]:
len(extracted_data)

637

In [12]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: list[Document]) -> List[Document]:
    """ 
    given a list of document objects, return a new list of document objects
    containing only 'source' in metadata and the original page_content.
    """

    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
                           
        )
    return minimal_docs    



In [15]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [16]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap =20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [17]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5859


In [ ]:
texts_chunk

In [18]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """ 
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding =download_embeddings()

C:\Users\IQRA TRADERS\AppData\Local\Temp\ipykernel_6940\4076849015.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [ ]:
embedding

In [ ]:
vector = embedding.embed_query("Hello World")
vector

In [22]:
print( "Vector Length:", len(vector))


Vector Length: 384


In [23]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [24]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [25]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [26]:
pc

In [27]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 384,
        metric = "cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)    


In [28]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [29]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [31]:
dswith = Document(
    page_content="Arman is a passionate AI-Engineer which is learning Agentic AI. ",
    metadata={"source" : "LinkedIn"}
)

In [32]:
docsearch.add_documents([dswith])

['8e03b108-bc66-4827-ad70-9977b351ac55']

In [38]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [39]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='f6082bd6-9158-441a-858f-e333597a4977', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='db1a00ad-5b69-4d1c-a19d-0e1a52331625', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='64a130ce-4fa2-4a39-8126-d133777a94a7', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged wi

In [40]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(model="gemini-3.7-flash")

c:\Users\IQRA TRADERS\miniconda3\envs\medibot\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\IQRA TRADERS\miniconda3\envs\medibot\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.ai.generativelanguage_v1beta once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.ai.generativelanguage_v1beta past that date.
  warnings.warn(message, FutureWarning)


In [41]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [42]:
system_prompt=(
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [ 
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [43]:
question_answer_chain = create_stuff_documents_chain(chatModel,prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
                                                    

In [50]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Date': 'Tue, 29 Sep 2026 19:03:53 GMT', 'Content-Type': 'application/json', 'Content-Length': '78', 'Connection': 'keep-alive', 'x-pinecone-auth-rejected-reason': 'Malformed domain', 'www-authenticate': 'Malformed domain', 'server': 'envoy'})
HTTP response body: {"error":{"code":"UNAUTHENTICATED","message":"Malformed domain"},"status":401}


In [ ]:
response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Date': 'Tue, 29 Sep 2026 19:03:37 GMT', 'Content-Type': 'application/json', 'Content-Length': '78', 'Connection': 'keep-alive', 'x-pinecone-auth-rejected-reason': 'Malformed domain', 'www-authenticate': 'Malformed domain', 'server': 'envoy'})
HTTP response body: {"error":{"code":"UNAUTHENTICATED","message":"Malformed domain"},"status":401}


In [ ]:
response = rag_chain.invoke({"input": "What is the treatment for Acne?"})
print(response["answer"])

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Date': 'Tue, 29 Sep 2026 19:03:34 GMT', 'Content-Type': 'application/json', 'Content-Length': '78', 'Connection': 'keep-alive', 'x-pinecone-auth-rejected-reason': 'Malformed domain', 'www-authenticate': 'Malformed domain', 'server': 'envoy'})
HTTP response body: {"error":{"code":"UNAUTHENTICATED","message":"Malformed domain"},"status":401}


In [ ]:
response = rag_chain.invoke({"input": "What is Arman learning?"})
print(response["answer"])

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Date': 'Tue, 29 Sep 2026 19:03:38 GMT', 'Content-Type': 'application/json', 'Content-Length': '78', 'Connection': 'keep-alive', 'x-pinecone-auth-rejected-reason': 'Malformed domain', 'www-authenticate': 'Malformed domain', 'server': 'envoy'})
HTTP response body: {"error":{"code":"UNAUTHENTICATED","message":"Malformed domain"},"status":401}
